# 02 - 网格特征粗分析

`X` 为 `[N, C, H, W]`（默认 **`C=31`**：16 bank + 15 inventory，`H×W=128×32`，`spatial_heatmap`）。将每个样本的网格展平为长度 `C×H×W` 的向量，对**展平下标**在正/负样本上统计均值与分位数（高维概览，非单 cell 解释）。默认配置下 inventory 已拼入 `X`，**无**独立 `tabular.npy`；本 Notebook 仅分析网格 `X`。

**数据源**：优先 `data/processed/X.npy`；若不存在则用 `data/splits/X_train.npy`（已按训练集对 `(C,H,W)` 标准化，与全量 `processed` 的尺度可能不同）。

**自动生成**：若上述均不存在，按 `PREPROCESS_EXP_YAML` 执行与 `bash scripts/preprocess.sh` 相同的预处理。**前提**：`configs/data.yaml` 中的 `mcelog_path`、`tickets_path`、`index_features_path`（及已配置的 `inventory_path`）在磁盘上必须存在。默认 `index_features_path` 指向 `data/index_stubs/features_5m_index_demo.csv`；全量请改为 `data/raw/features_index_h300s_step300s_full.csv` 等（先 `bash scripts/build_index.sh full`），否则会预检失败。

**Notebook 提示**：配置校验使用 **`data.window_seconds`**。若单元格输出与当前源码不符，请 **Kernel → Restart** 后重新执行，或使用「清除所有输出」后再跑。

**输出**：`outputs/tables/feature_summary_by_label.csv`、`outputs/figures/feature_example_pos_neg.png`

**流水线关系**：特征来自 **preprocess**；端到端冒烟/全量请用 **`04_run_pipelines.ipynb`**（本 Notebook 适合 preprocess 后检查 `X` 是否合理）。


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
_SRC = str(PROJECT_ROOT / "src")
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)
for _name in list(sys.modules):
    if _name == "dram2cnn" or _name.startswith("dram2cnn."):
        del sys.modules[_name]

# 缺少预处理产物时自动跑预处理；与当前分析目标对齐（horizon 实验请改为 exp_horizon_30m.yaml 等）
PREPROCESS_EXP_YAML = PROJECT_ROOT / "configs" / "experiments" / "exp_baseline.yaml"

OUT_T = PROJECT_ROOT / "outputs" / "tables"
OUT_F = PROJECT_ROOT / "outputs" / "figures"
OUT_T.mkdir(parents=True, exist_ok=True)
OUT_F.mkdir(parents=True, exist_ok=True)


In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

proc = PROJECT_ROOT / "data" / "processed"
spl = PROJECT_ROOT / "data" / "splits"


def _have_processed() -> bool:
    return (proc / "X.npy").is_file() and (proc / "y.npy").is_file()


def _have_splits_train() -> bool:
    return (spl / "X_train.npy").is_file() and (spl / "y_train.npy").is_file()


from dram2cnn.utils.preprocess_preflight import check_preprocess_raw_inputs


if not _have_processed() and not _have_splits_train():
    cfg_path = PREPROCESS_EXP_YAML.resolve()
    if not PREPROCESS_EXP_YAML.is_file():
        raise FileNotFoundError(f"预处理配置文件不存在: {PREPROCESS_EXP_YAML}")
    check_preprocess_raw_inputs(PREPROCESS_EXP_YAML)
    print(
        "提示：未找到预处理/切分特征文件，将自动运行预处理（与 bash scripts/preprocess.sh 相同）。\n"
        f"配置: {cfg_path}\n"
        "原始表较大时可能耗时较久，请等待完成。\n"
    )
    from dram2cnn.pipelines.run_preprocess import main as run_preprocess_main

    run_preprocess_main(str(cfg_path))

if _have_processed():
    X_path, y_path = proc / "X.npy", proc / "y.npy"
    print("数据源: data/processed/（全量池，未按 train 重标准化）")
    try:
        from dram2cnn.utils.config import load_experiment_config
        from dram2cnn.utils.experiment_guards import assert_processed_matches_experiment

        _pcfg = load_experiment_config(PREPROCESS_EXP_YAML)
        assert_processed_matches_experiment(_pcfg, PROJECT_ROOT)
    except (ValueError, FileNotFoundError) as _e:
        print(f"警告：processed 与 PREPROCESS_EXP_YAML 可能不一致（{_e}）；请改 PREPROCESS_EXP_YAML 或重跑 preprocess。")
elif _have_splits_train():
    X_path, y_path = spl / "X_train.npy", spl / "y_train.npy"
    print(
        "数据源: data/splits/ 训练集（未找到 processed/X.npy；此处为 splitter 按 train 统计量标准化后的 X_train）"
    )
else:
    raise RuntimeError(
        "自动预处理后仍未找到 data/processed/X.npy、y.npy 或 data/splits/X_train.npy、y_train.npy；请检查 data/raw/ 与上方报错。"
    )

X = np.load(X_path, mmap_mode="r")
y = np.load(y_path, mmap_mode="r")
print("X.shape", X.shape, "y.shape", y.shape)
print("pos rate", float(y.mean()))

if X.ndim != 4:
    raise ValueError(f"期望 X 为 [N,C,H,W]，实际 {X.shape}")

N, C, H, W = X.shape
print("C, H, W =", int(C), int(H), int(W))

# 与 model.yaml / preprocess_manifest 对照（spatial 默认 31×128×32）
import json
import yaml

model_p = PROJECT_ROOT / "configs" / "model.yaml"
if model_p.is_file():
    mc = yaml.safe_load(model_p.read_text(encoding="utf-8"))
    exp = (int(mc["in_channels"]), int(mc["grid_height"]), int(mc["grid_width"]))
    if (int(C), int(H), int(W)) != exp:
        print(f"警告：X (C,H,W)={(int(C), int(H), int(W))} 与 model.yaml {exp} 不一致 → 请重跑 preprocess")
    else:
        print(f"与 model.yaml 一致：{exp}")

manifest_p = PROJECT_ROOT / "data" / "processed" / "preprocess_manifest.json"
if manifest_p.is_file():
    man = json.loads(manifest_p.read_text(encoding="utf-8"))
    print(
        "manifest:",
        "grid_shape=", man.get("grid_shape"),
        "horizon=", man.get("label_horizon_seconds"),
        "n_empty_grids=", man.get("n_empty_grids", 0),
    )
    if int(man.get("n_empty_grids") or 0) > 0:
        print("提示：部分样本 mcelog 窗口为空（全零网格），请检查索引 sid 与 raw 对齐")

X2 = X.reshape(N, C * H * W)
print("展平维数", int(C * H * W))

rng = np.random.RandomState(42)
MAX_N = 50_000
idx = np.arange(N)
if N > MAX_N:
    idx = rng.choice(N, size=MAX_N, replace=False)
    Xs = np.asarray(X2[idx], dtype=np.float64)
    ys = np.asarray(y[idx])
else:
    Xs = np.asarray(X2, dtype=np.float64)
    ys = np.asarray(y)

rows = []
for fi in range(Xs.shape[1]):
    for lab in (0, 1):
        m = ys == lab
        if m.sum() == 0:
            continue
        col = Xs[m, fi]
        rows.append({
            "feature_index": fi,
            "label": lab,
            "mean": float(col.mean()),
            "std": float(col.std()),
            "p01": float(np.percentile(col, 1)),
            "p99": float(np.percentile(col, 99)),
        })
pd.DataFrame(rows).to_csv(OUT_T / "feature_summary_by_label.csv", index=False)
print("已写入", OUT_T / "feature_summary_by_label.csv")


数据源: data/processed/（全量池，未按 train 重标准化）
X.shape (500, 31, 128, 32) y.shape (500,)
pos rate 0.004
C, H, W = 31 128 32
与 model.yaml 一致：(31, 128, 32)
manifest: grid_shape= [31, 128, 32] horizon= 300.0 n_empty_grids= 228
提示：部分样本 mcelog 窗口为空（全零网格），请检查索引 sid 与 raw 对齐
展平维数 126976
已写入 /root/workspace/outputs/tables/feature_summary_by_label.csv


In [3]:
import matplotlib.pyplot as plt

# 取若干展平下标画直方图（正负样本）；下标 0 / 中位 / 末位
ndeflat = int(C * H * W)
mid = max(0, (ndeflat - 1) // 2) if ndeflat > 0 else 0
feat_ids = sorted({0, mid, max(0, ndeflat - 1)})
fig, axes = plt.subplots(1, len(feat_ids), figsize=(4 * len(feat_ids), 3))
if len(feat_ids) == 1:
    axes = [axes]
for ax, fi in zip(axes, feat_ids):
    for lab, color in ((0, "steelblue"), (1, "darkorange")):
        m = ys == lab
        if m.sum() == 0:
            continue
        ax.hist(Xs[m, fi], bins=40, density=True, alpha=0.45, color=color, label=f"y={lab}")
    ax.set_title(f"feature {fi}")
    ax.legend()
plt.tight_layout()
fig.savefig(OUT_F / "feature_example_pos_neg.png", dpi=120)
plt.close()
print("已写入", OUT_F / "feature_example_pos_neg.png")


已写入 /root/workspace/outputs/figures/feature_example_pos_neg.png
